# VGG by PyTorch (NG)

Source : https://pytorch.org/hub/pytorch_vision_vgg/ dont les performances Top-1 et Top-5.

Adaptation et mise à jour de l'exemple de PyTorch pour ce modèle.

modules

In [1]:
import os
from typing import Tuple, Callable
from PIL import Image
import torch
import torchvision
from torchvision import transforms as T
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

from datasets import DATASET_1, DATASET_2, CustomImageDataset, get_label_data_from_filename

device

In [2]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} device available")

mps device available


modèle

In [3]:
#model_by_pytorch = torch.hub.load('pytorch/vision:v0.10.0', 'vgg16', pretrained=True)
MODEL_NAME = 'vgg16'
MODEL_WEIGHTS = torchvision.models.VGG16_Weights
model_by_pytorch = torch.hub.load('pytorch/vision', MODEL_NAME, weights=MODEL_WEIGHTS.IMAGENET1K_V1)

Using cache found in /Users/me/.cache/torch/hub/pytorch_vision_main


Chargement du dataset

In [4]:
"""
transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
"""

transforms = MODEL_WEIGHTS.IMAGENET1K_V1.transforms()

DATASET = DATASET_1
get_label_data = lambda f: get_label_data_from_filename(f, DATASET["path"])
dataset_path = DATASET["mounted_path"] if os.path.exists(DATASET["mounted_path"]) else DATASET["path"]
print("Using Dataset (name, path)", DATASET["name"], dataset_path)

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=True,
    only_label_idx=True,
    get_label_data=get_label_data,
    read_image_with="PIL",
    )

Using Dataset (name, path) imagenet_val_images (50K images) /Users/me/Documents/Work/Dev/_data/imagenet_val_images


In [5]:
print(transforms)

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [6]:
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [7]:
all_predictions_batch = []
all_expecteds_batch = []
model_by_pytorch.eval()
model_by_pytorch.to(device)
for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    output = model_by_pytorch.forward(batch[0].to(device)) # To carefully transfer to CPU before append in list
    output = output.to("cpu").detach()
    probabilities = torch.nn.functional.softmax(output, dim=1)
    top5_prob, top5_catid = torch.topk(probabilities, 5)
    all_predictions_batch.append((top5_prob.tolist(), top5_catid.tolist()))
    all_expecteds_batch.append(batch[1].tolist())

  0%|          | 0/1563 [00:00<?, ?it/s]

In [8]:
from collections import Counter
distribution_computed_top1 = Counter()
distribution_computed_top5 = Counter()
distribution_expected = Counter()

right_top1 = 0
right_top5 = 0

for (predictions_batch_prob, predictions_batch_catid), expected_batch in zip(all_predictions_batch, all_expecteds_batch):
    predictions_batch_catid = torch.tensor(predictions_batch_catid)
    expected_batch = torch.tensor(expected_batch).reshape(-1, 1)
    expected_in_top1 = (predictions_batch_catid[:, 0] - expected_batch) == 0
    expected_in_top5 = (predictions_batch_catid - expected_batch) == 0

    right_top1 += expected_in_top1.sum().item()
    right_top5 += expected_in_top5.sum().item()

    distribution_computed_top1.update(predictions_batch_catid[:, 0].ravel().tolist())
    distribution_computed_top5.update(predictions_batch_catid.ravel().tolist())
    distribution_expected.update(expected_batch.ravel().tolist())

In [9]:
print(f"Justesse top1: {right_top1} ({right_top1 / len(dataset) * 100:.2f}%) | Error: {100. - right_top1 / len(dataset) * 100:.2f}%")
print(f"Justesse top5: {right_top5} ({right_top5 / len(dataset) * 100:.2f}%) | Error: {100. - right_top5 / len(dataset) * 100:.2f}%")

Justesse top1: 37257 (74.51%) | Error: 25.49%
Justesse top5: 45195 (90.39%) | Error: 9.61%


In [10]:
print(distribution_computed_top1.total())
print(distribution_computed_top5.total())

50000
250000


In [11]:
distribution_computed_top1.most_common(10)

[(679, 96),
 (527, 82),
 (819, 81),
 (709, 78),
 (926, 77),
 (474, 74),
 (162, 73),
 (404, 71),
 (526, 71),
 (382, 71)]

In [12]:
distribution_computed_top5.most_common(10)

[(60, 704),
 (754, 610),
 (380, 576),
 (973, 557),
 (536, 554),
 (134, 528),
 (54, 515),
 (200, 498),
 (663, 497),
 (911, 496)]

In [13]:
list(distribution_expected.items())[:10]

[(55, 50),
 (404, 50),
 (989, 50),
 (503, 50),
 (518, 50),
 (516, 50),
 (343, 50),
 (436, 50),
 (898, 50),
 (290, 50)]